# Flight Delay Prediction — XGBoost


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score,
    precision_recall_curve, RocCurveDisplay, average_precision_score
)
from imblearn.over_sampling import SMOTE
import joblib

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

RANDOM_STATE  = 42
TEST_SIZE     = 0.20
OPTUNA_TRIALS = 80
CV_FOLDS      = 5
EARLY_STOP    = 50
np.random.seed(RANDOM_STATE)
print("Libraries loaded ✅")

## 2. Load Data

In [ ]:
train = pd.read_csv("flight_dataset_v1.csv", parse_dates=[
    "scheduled_departure_time", "actual_departure_time",
    "scheduled_arrival_time", "actual_arrival_time", "dep_date_only"
])
print(f"Shape: {train.shape}")
print(train.head())

## 3. Remove Cancelled & Diverted

In [ ]:
print(f"Cancelled : {train['is_cancelled'].sum()} rows")
print(f"Diverted  : {train['is_diverted'].sum()} rows")

df = train[
    (train['is_cancelled'] == 0) &
    (train['is_diverted']  == 0)
].copy().reset_index(drop=True)

df['departure_delay_min'] = df['departure_delay_min'].clip(lower=0)
print(f"Clean shape: {df.shape}")

## 4. Feature Engineering

In [ ]:
# ── Targets ───────────────────────────────────────────────────────────────────
df['target_binary']   = (df['departure_delay_min'] >= 15).astype(int)
df['target_severity'] = (df['departure_delay_min'] >= 60).astype(int)

print(f"Binary   — On-Time: {(df['target_binary']==0).sum():,} | Delayed: {(df['target_binary']==1).sum():,}")
delayed_df = df[df['target_binary']==1]
print(f"Severity — Minor  : {(delayed_df['target_severity']==0).sum():,} | Major: {(delayed_df['target_severity']==1).sum():,}")

# ── Cyclical time features ─────────────────────────────────────────────────────
for col, prefix in [('scheduled_departure_hour','dep'),('scheduled_arrival_hour','arr')]:
    if col in df.columns:
        df[f'{prefix}_hour_sin'] = np.sin(2 * np.pi * df[col] / 24)
        df[f'{prefix}_hour_cos'] = np.cos(2 * np.pi * df[col] / 24)

# ── Day of week: string → int → cyclical ──────────────────────────────────────
day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
           'Friday':4,'Saturday':5,'Sunday':6}
for col in ['scheduled_departure_day_of_week','scheduled_arrival_day_of_week']:
    if col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].map(day_map)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / 7)
        df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / 7)

# ── Interaction features ───────────────────────────────────────────────────────
if all(c in df.columns for c in ['dep_precipitation','arr_precipitation']):
    df['total_precipitation'] = df['dep_precipitation'] + df['arr_precipitation']
    df['any_precipitation']   = ((df['dep_precipitation']>0)|(df['arr_precipitation']>0)).astype(int)

if all(c in df.columns for c in ['scheduled_late_night_departure','dep_is_monsoon_season']):
    df['night_monsoon'] = df['scheduled_late_night_departure'] * df['dep_is_monsoon_season']

if all(c in df.columns for c in ['airline_delay_rate','route_delay_rate']):
    df['airline_route_risk'] = df['airline_delay_rate'] * df['route_delay_rate']

if all(c in df.columns for c in ['route_avg_delay','scheduled_is_peak_hour']):
    df['route_delay_peak'] = df['route_avg_delay'] * df['scheduled_is_peak_hour']

print("Feature engineering complete ✅")

## 5. Drop Leakage & Encode

In [ ]:
DROP_COLS = [
    'id', 'dep_date_only',
    'actual_departure_time', 'actual_arrival_time',
    'scheduled_departure_time', 'scheduled_arrival_time',
    'departure_delay_min', 'arrival_delay_min', 'delay_ratio',
    'binary_delay_dep', 'binary_delay_arr',
    'delay_class_dep', 'delay_class_arr',
    'is_diverted', 'is_cancelled',
    'origin_code',
    'dep_wind_speed_100m', 'arr_wind_speed_100m',
    'target_binary', 'target_severity',
]

CAT_FEATURES = ['airline', 'aircraft_type', 'destination_code', 'route']

feature_cols = [c for c in df.columns if c not in DROP_COLS]
dt_cols = df[feature_cols].select_dtypes(include=['datetime64','object']).columns.tolist()

# Label encode categoricals (required for RF, XGB, LR — unlike CatBoost)
le_dict = {}
for col in CAT_FEATURES:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

feature_cols = [c for c in feature_cols if c not in dt_cols]
X_all      = df[feature_cols].copy()
y_binary   = df['target_binary'].copy()
y_severity = df['target_severity'].copy()

print(f"Feature matrix : {X_all.shape}")
print(f"Binary target  : {dict(y_binary.value_counts().sort_index())}")

## 6. Model 1 — Binary Classifier
### 6a. Split

In [ ]:
X1_train, X1_valid, y1_train, y1_valid = train_test_split(
    X_all, y_binary,
    test_size=TEST_SIZE, stratify=y_binary, random_state=RANDOM_STATE
)
print(f"Train: {X1_train.shape} | Valid: {X1_valid.shape}")
print(f"Class dist: {dict(y1_train.value_counts().sort_index())}")

### 6b. SMOTE

In [ ]:
sm1 = SMOTE(random_state=RANDOM_STATE)
X1_train_res, y1_train_res = sm1.fit_resample(X1_train, y1_train)
print(f"Before SMOTE: {dict(y1_train.value_counts().sort_index())}")
print(f"After  SMOTE: {dict(pd.Series(y1_train_res).value_counts().sort_index())}")

### 6c. Optuna Hyperparameter Tuning

In [ ]:
scale_pos_weight = y1_train.value_counts()[0] / y1_train.value_counts()[1]
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

def m1_objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth':          trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 10),
        'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel':  trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'gamma':              trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight':   scale_pos_weight,
        'eval_metric':        'auc',
        'use_label_encoder':  False,
        'random_state':       RANDOM_STATE,
        'n_jobs':             -1,
        'verbosity':          0,
    }
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(
        xgb.XGBClassifier(**params), X1_train_res, y1_train_res,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

print(f"Running Optuna — {OPTUNA_TRIALS} trials...")
study1 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study1.optimize(m1_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nBest CV ROC AUC : {study1.best_value:.4f}")
for k, v in study1.best_params.items():
    print(f"  {k}: {v}")

### 6d. Train Final Model 1 with Early Stopping

In [ ]:
MODEL_NAME = "XGBoost"

best_p1 = study1.best_params.copy()
model1 = xgb.XGBClassifier(
    **best_p1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=EARLY_STOP,
    n_estimators=3000,   # ceiling — early stopping controls actual count
)

model1.fit(
    X1_train_res, y1_train_res,
    eval_set=[(X1_valid, y1_valid)],
    verbose=100
)

y1_proba = model1.predict_proba(X1_valid)[:, 1]
print(f"Best iteration   : {model1.best_iteration}")
print(f"Validation AUC   : {roc_auc_score(y1_valid, y1_proba):.4f}")

### 6e. Threshold Tuning

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y1_valid, y1_proba)
f1s        = 2*(precisions[:-1]*recalls[:-1]) / (precisions[:-1]+recalls[:-1]+1e-8)
best_idx   = np.argmax(f1s)
threshold1 = thresholds[best_idx]

print(f"Default (0.50) → F1: {f1_score(y1_valid,(y1_proba>=0.50).astype(int)):.4f}")
print(f"Tuned  ({threshold1:.4f}) → F1: {f1s[best_idx]:.4f}")

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(thresholds, precisions[:-1], label='Precision', color='#3498db', lw=2)
ax.plot(thresholds, recalls[:-1],    label='Recall',    color='#e67e22', lw=2)
ax.plot(thresholds, f1s,             label='F1',        color='#2ecc71', lw=2)
ax.axvline(threshold1, color='red',  linestyle='--', lw=1.5, label=f'Best={threshold1:.3f}')
ax.axvline(0.50,       color='gray', linestyle=':',  lw=1.5, label='Default=0.50')
ax.set_xlabel('Threshold'); ax.set_title('Model 1 — Threshold Tuning', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

y1_pred_tuned = (y1_proba >= threshold1).astype(int)

### 6f. Evaluation

In [ ]:
print("="*55)
print("  MODEL 1 — BINARY (XGBoost)")
print("="*55)
print(f"  Accuracy    : {accuracy_score(y1_valid, y1_pred_tuned):.4f}")
print(f"  ROC AUC     : {roc_auc_score(y1_valid, y1_proba):.4f}")
print(f"  F1 Macro    : {f1_score(y1_valid, y1_pred_tuned, average='macro'):.4f}")
print(f"  F1 Weighted : {f1_score(y1_valid, y1_pred_tuned, average='weighted'):.4f}")
print(f"  Precision   : {precision_score(y1_valid, y1_pred_tuned, zero_division=0):.4f}")
print(f"  Recall      : {recall_score(y1_valid, y1_pred_tuned, zero_division=0):.4f}")
print(classification_report(y1_valid, y1_pred_tuned, target_names=['On-Time','Delayed']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm1 = confusion_matrix(y1_valid, y1_pred_tuned)
ConfusionMatrixDisplay(cm1, display_labels=['On-Time','Delayed']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Model 1 Confusion Matrix (t={threshold1:.3f})', fontweight='bold')

RocCurveDisplay.from_predictions(y1_valid, y1_proba, ax=axes[1], name=MODEL_NAME)
axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].set_title('Model 1 — ROC Curve', fontweight='bold')
axes[1].legend()

ap1 = average_precision_score(y1_valid, y1_proba)
precisions_pr, recalls_pr, _ = precision_recall_curve(y1_valid, y1_proba)
axes[2].plot(recalls_pr, precisions_pr, color='#9b59b6', lw=2)
axes[2].fill_between(recalls_pr, precisions_pr, alpha=0.1, color='#9b59b6')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title(f'PR Curve (AP={ap1:.3f})', fontweight='bold')

plt.tight_layout(); plt.show()

### 6g. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,7))
for ax, imp_type in zip(axes, ['weight','gain']):
    imp = pd.Series(model1.get_booster().get_score(importance_type=imp_type))
    imp = imp.sort_values(ascending=False).head(20)
    imp.plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.85)
    ax.invert_yaxis()
    ax.set_title(f'XGBoost Feature Importance — {imp_type.capitalize()}', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Model 2 — Severity Classifier

In [ ]:
delayed_mask = y_binary == 1
X_delayed    = X_all[delayed_mask].copy().reset_index(drop=True)
y_sev        = y_severity[delayed_mask].copy().reset_index(drop=True)

print(f"Delayed subset : {X_delayed.shape[0]} rows")
print(f"Minor (0)      : {(y_sev==0).sum()} | Major (1): {(y_sev==1).sum()}")

X2_train, X2_valid, y2_train, y2_valid = train_test_split(
    X_delayed, y_sev,
    test_size=TEST_SIZE, stratify=y_sev, random_state=RANDOM_STATE
)

k = max(1, min(3, (y2_train==1).sum()-1))
sm2 = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
X2_train_res, y2_train_res = sm2.fit_resample(X2_train, y2_train)
print(f"After SMOTE: {dict(pd.Series(y2_train_res).value_counts().sort_index())}")

In [ ]:
spw2 = (y2_train==0).sum() / (y2_train==1).sum()

def m2_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight':  spw2,
        'eval_metric':       'auc',
        'use_label_encoder': False,
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
        'verbosity':         0,
    }
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(xgb.XGBClassifier(**params), X2_train_res, y2_train_res,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study2 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study2.optimize(m2_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print(f"Best CV AUC (M2): {study2.best_value:.4f}")

model2 = xgb.XGBClassifier(
    **study2.best_params, scale_pos_weight=spw2,
    eval_metric='auc', use_label_encoder=False,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    early_stopping_rounds=EARLY_STOP, n_estimators=3000
)
model2.fit(X2_train_res, y2_train_res, eval_set=[(X2_valid, y2_valid)], verbose=False)
y2_proba = model2.predict_proba(X2_valid)[:, 1]
print(f"Validation AUC: {roc_auc_score(y2_valid, y2_proba):.4f}")

In [ ]:
precisions2, recalls2, thresholds2 = precision_recall_curve(y2_valid, y2_proba)
f1s2       = 2*(precisions2[:-1]*recalls2[:-1]) / (precisions2[:-1]+recalls2[:-1]+1e-8)
best_idx2  = np.argmax(f1s2)
threshold2 = thresholds2[best_idx2]

print(f"Default (0.50) → F1: {f1_score(y2_valid,(y2_proba>=0.50).astype(int)):.4f}")
print(f"Tuned  ({threshold2:.4f}) → F1: {f1s2[best_idx2]:.4f}")
y2_pred_tuned = (y2_proba >= threshold2).astype(int)

In [ ]:
print("="*55)
print(f"  MODEL 2 — SEVERITY ({MODEL_NAME})")
print("="*55)
print(f"  Accuracy  : {accuracy_score(y2_valid, y2_pred_tuned):.4f}")
print(f"  ROC AUC   : {roc_auc_score(y2_valid, y2_proba):.4f}")
print(f"  F1 Macro  : {f1_score(y2_valid, y2_pred_tuned, average='macro'):.4f}")
print(classification_report(y2_valid, y2_pred_tuned, target_names=['Minor','Major']))

fig, axes = plt.subplots(1,2,figsize=(13,5))
cm2 = confusion_matrix(y2_valid, y2_pred_tuned)
ConfusionMatrixDisplay(cm2, display_labels=['Minor','Major']).plot(ax=axes[0], cmap='Oranges', colorbar=False)
axes[0].set_title(f'Model 2 Confusion Matrix (t={threshold2:.3f})', fontweight='bold')
RocCurveDisplay.from_predictions(y2_valid, y2_proba, ax=axes[1], name=MODEL_NAME)
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('Model 2 — ROC Curve', fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Cascaded Prediction

In [ ]:
def cascaded_predict(X_new, model_b, model_s, thresh_b, thresh_s):
    prob_del  = model_b.predict_proba(X_new)[:, 1]
    is_del    = (prob_del >= thresh_b).astype(int)
    sev_prob  = np.zeros(len(X_new))
    del_idx   = np.where(is_del == 1)[0]
    if len(del_idx) > 0:
        sev_prob[del_idx] = model_s.predict_proba(X_new.iloc[del_idx])[:, 1]
    is_major   = (sev_prob >= thresh_s).astype(int)
    final_pred = np.where(is_del==0, 0, np.where(is_major==0, 1, 2))
    label_map  = {0:'On-Time (<15 min)', 1:'Minor (15–59 min)', 2:'Major (≥60 min)'}
    confidence = np.where(is_del==0, 1-prob_del,
                 np.where(is_major==1, sev_prob, 1-sev_prob))
    return pd.DataFrame({
        'delay_predicted': final_pred,
        'delay_label':     [label_map[p] for p in final_pred],
        'binary_prob':     prob_del.round(4),
        'severity_prob':   sev_prob.round(4),
        'confidence':      confidence.round(4)
    })

y_true_3class = np.where(
    y_binary[X1_valid.index]==0, 0,
    np.where(y_severity[X1_valid.index]==0, 1, 2)
)
results = cascaded_predict(X1_valid, model1, model2, threshold1, threshold2)

print("="*55)
print(f"  CASCADED SYSTEM — END-TO-END ({MODEL_NAME})")
print("="*55)
print(f"  Accuracy (3-class) : {accuracy_score(y_true_3class, results['delay_predicted']):.4f}")
print(f"  F1 Macro           : {f1_score(y_true_3class, results['delay_predicted'], average='macro'):.4f}")
print(f"  F1 Weighted        : {f1_score(y_true_3class, results['delay_predicted'], average='weighted'):.4f}")
print(classification_report(y_true_3class, results['delay_predicted'],
      target_names=['On-Time','Minor','Major']))

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(
    confusion_matrix(y_true_3class, results['delay_predicted']),
    display_labels=['On-Time','Minor','Major']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Cascaded {MODEL_NAME} — End-to-End', fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Optuna Analysis

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,5))
try:
    optuna.visualization.matplotlib.plot_optimization_history(study1, ax=axes[0])
    axes[0].set_title('Optuna History — Model 1', fontweight='bold')
    optuna.visualization.matplotlib.plot_param_importances(study1, ax=axes[1])
    axes[1].set_title('Hyperparameter Importance — Model 1', fontweight='bold')
except Exception as e:
    print(f"Plot note: {e}")
plt.tight_layout(); plt.show()

## 10. Results Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': f'{MODEL_NAME} M1 (Binary)',   'ROC AUC': f"{roc_auc_score(y1_valid,y1_proba):.4f}",
     'Accuracy': f"{accuracy_score(y1_valid,y1_pred_tuned):.4f}",
     'F1 Macro': f"{f1_score(y1_valid,y1_pred_tuned,average='macro'):.4f}",
     'F1 Weighted': f"{f1_score(y1_valid,y1_pred_tuned,average='weighted'):.4f}",
     'Threshold': f"{threshold1:.4f}", 'Leakage': 'None'},
    {'Model': f'{MODEL_NAME} M2 (Severity)', 'ROC AUC': f"{roc_auc_score(y2_valid,y2_proba):.4f}",
     'Accuracy': f"{accuracy_score(y2_valid,y2_pred_tuned):.4f}",
     'F1 Macro': f"{f1_score(y2_valid,y2_pred_tuned,average='macro'):.4f}",
     'F1 Weighted': f"{f1_score(y2_valid,y2_pred_tuned,average='weighted'):.4f}",
     'Threshold': f"{threshold2:.4f}", 'Leakage': 'None'},
    {'Model': f'{MODEL_NAME} Cascaded',      'ROC AUC': '—',
     'Accuracy': f"{accuracy_score(y_true_3class,results['delay_predicted']):.4f}",
     'F1 Macro': f"{f1_score(y_true_3class,results['delay_predicted'],average='macro'):.4f}",
     'F1 Weighted': f"{f1_score(y_true_3class,results['delay_predicted'],average='weighted'):.4f}",
     'Threshold': f"M1={threshold1:.3f}/M2={threshold2:.3f}", 'Leakage': 'None'},
])
print(summary.T.to_string(header=False))
summary.to_csv(f'{MODEL_NAME.lower().replace(" ","_")}_results.csv', index=False)
print(f"\nSaved: {MODEL_NAME.lower().replace(' ','_')}_results.csv")

## 11. Save Models

In [ ]:
model1.save_model('xgb_model1_binary.json')
model2.save_model('xgb_model2_severity.json')
joblib.dump(le_dict, 'xgb_label_encoders.pkl')
joblib.dump({'threshold_binary': threshold1, 'threshold_severity': threshold2,
             'feature_cols': list(X_all.columns)}, 'xgb_config.pkl')
print("Saved: xgb_model1_binary.json | xgb_model2_severity.json | xgb_config.pkl")